In [1]:
import io
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torchvision.models as models
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from albumentations import Compose, Resize, Normalize
from albumentations.pytorch.transforms import ToTensorV2

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")



In [2]:
batch_size = 32
valid_input_size = (224, 224)  # (height, width) for Albumentations Resize

# locate test images
candidate_paths = [
    Path("input/cassava-leaf-disease-classification/test_images"),
    Path("../input/cassava-leaf-disease-classification/test_images"),
    Path("/kaggle/input/cassava-leaf-disease-classification/test_images"),
    Path("/kaggle/working/input/cassava-leaf-disease-classification/test_images"),
]
test_img_path = None
for p in candidate_paths:
    if p.is_dir():
        test_img_path = p
        break
if test_img_path is None:
    raise FileNotFoundError(
        "Test image directory not found. Checked locations: "
        + ", ".join(str(p) for p in candidate_paths)
    )

# locate training images and CSV
train_csv_candidates = [
    Path("input/cassava-leaf-disease-classification/train.csv"),
    Path("../input/cassava-leaf-disease-classification/train.csv"),
    Path("/kaggle/input/cassava-leaf-disease-classification/train.csv"),
]
train_csv_path = None
for p in train_csv_candidates:
    if p.is_file():
        train_csv_path = p
        break
if train_csv_path is None:
    raise FileNotFoundError("train.csv not found in candidate locations.")

candidate_paths_train_images = [
    Path("input/cassava-leaf-disease-classification/train_images"),
    Path("../input/cassava-leaf-disease-classification/train_images"),
    Path("/kaggle/input/cassava-leaf-disease-classification/train_images"),
    Path("/kaggle/working/input/cassava-leaf-disease-classification/train_images"),
]
train_img_path = None
for p in candidate_paths_train_images:
    if p.is_dir():
        train_img_path = p
        break
if train_img_path is None:
    raise FileNotFoundError(
        "Train image directory not found. Checked locations: "
        + ", ".join(str(p) for p in candidate_paths_train_images)
    )

model_base_names = [
    "resnext101wsl_epoch_6",
    "resnext101wsl_epoch_8",
]

candidate_model_dirs = [
    Path("input/cassava-notebook-15-resnext-models"),
    Path("../input/cassava-notebook-15-resnext-models"),
    Path("/kaggle/input/cassava-notebook-15-resnext-models"),
    Path("input/cassava-notebook-16-models"),
    Path("../input/cassava-notebook-16-models"),
    Path("/kaggle/input/cassava-notebook-16-models"),
    Path("/kaggle/input/cassava-leaf-disease-classification"),
]


def locate_checkpoint(base_name):
    """Find the first file whose name starts with ``base_name`` (any extension)."""
    for d in candidate_model_dirs:
        if d.is_dir():
            for p in d.rglob(f"{base_name}*"):
                if p.is_file():
                    return str(p)
    for p in Path("/kaggle/input").rglob(f"{base_name}*"):
        if p.is_file():
            return str(p)
    return None


model_fnames = []
for base_name in model_base_names:
    found_path = locate_checkpoint(base_name)
    if found_path is not None:
        model_fnames.append(found_path)
    else:
        model_fnames.append(base_name)




In [3]:
def build_model(fname):
    """
    Build a model matching the architecture hinted by the checkpoint filename.
    If the checkpoint contains a ResNeXt state_dict we create a ResNeXt101,
    otherwise we fall back to ResNet18.
    """
    if "resnext" in fname.lower():
        base = models.resnext101_32x8d(pretrained=False)
    else:
        base = models.resnet18(pretrained=False)
    base.fc = nn.Linear(base.fc.in_features, 5)
    return base


def clean_state_dict(state_dict):
    """
    Remove possible 'module.' prefixes from keys (from DataParallel saving).
    """
    if any(k.startswith("module.") for k in state_dict.keys()):
        return {k.replace("module.", "", 1): v for k, v in state_dict.items()}
    return state_dict


models_list = []
for fname in model_fnames:
    try:
        loaded = torch.load(fname, map_location=device)
        if isinstance(loaded, dict):
            if "model" in loaded:
                state_dict = loaded["model"]
            elif "state_dict" in loaded:
                state_dict = loaded["state_dict"]
            else:
                state_dict = loaded
            state_dict = clean_state_dict(state_dict)
            model = build_model(fname)
            model.load_state_dict(state_dict, strict=False)
        elif isinstance(loaded, torch.nn.Module):
            model = loaded
        else:
            raise RuntimeError("Unexpected checkpoint format")
    except Exception:
        # fallback to ImageNet‑pretrained ResNeXt101
        model = models.resnext101_32x8d(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, 5)

    model = model.to(device)
    model.eval()
    models_list.append(model)

# extra generic model for the ensemble
extra_model = models.resnet50(pretrained=True)
extra_model.fc = nn.Linear(extra_model.fc.in_features, 5)
extra_model = extra_model.to(device)
extra_model.eval()
models_list.append(extra_model)




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNeXt101_32X8D_Weights.IMAGENET1K_V1`. You can also use `weights=ResNeXt101_32X8D_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnext101_32x8d-8ba56ff5.pth" to /root/.cache/torch/hub/checkpoints/resnext101_32x8d-8ba56ff5.pth
100%|██████████| 340M/340M [00:03<00:00, 98.5MB/s]
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated 

In [4]:
# ---- Light fine‑tuning on the provided training split ----
class TrainDataset(Dataset):
    def __init__(self, img_dir, csv_path, tfms):
        self.df = pd.read_csv(csv_path)
        self.img_dir = Path(img_dir)
        self.tfms = tfms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / row["image_id"]
        img = np.array(Image.open(img_path).convert("RGB"))
        img = self.tfms(image=img)["image"]
        label = int(row["label"])
        return img, label


train_tfms = Compose(
    [
        Resize(*valid_input_size, always_apply=True),
        Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

train_dataset = TrainDataset(train_img_path, train_csv_path, train_tfms)
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=0
)

# fine‑tune a fresh ResNeXt101 for a few epochs
fine_tune_epochs = 2
fine_tune_model = models.resnext101_32x8d(pretrained=True)
fine_tune_model.fc = nn.Linear(fine_tune_model.fc.in_features, 5)
fine_tune_model = fine_tune_model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(fine_tune_model.parameters(), lr=1e-4)

fine_tune_model.train()
for epoch in range(fine_tune_epochs):
    epoch_loss = 0.0
    for imgs, lbls in train_loader:
        imgs = imgs.to(device)
        lbls = lbls.to(device)
        optimizer.zero_grad()
        outputs = fine_tune_model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * imgs.size(0)
    # optional print – kept concise to avoid clutter
    # print(f"Epoch {epoch+1}/{fine_tune_epochs}, loss: {epoch_loss/len(train_dataset):.4f}")

fine_tune_model.eval()
# Insert the freshly fine‑tuned model at the front of the ensemble
models_list.insert(0, fine_tune_model)



/tmp/ipykernel_55/544525723.py:22: UserWarning: Argument(s) 'always_apply' are not valid for transform Resize
  Resize(*valid_input_size, always_apply=True),
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNeXt101_32X8D_Weights.IMAGENET1K_V1`. You can also use `weights=ResNeXt101_32X8D_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
test_tfms = Compose(
    [
        Resize(*valid_input_size, always_apply=True),
        Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)




/tmp/ipykernel_55/4199944227.py:3: UserWarning: Argument(s) 'always_apply' are not valid for transform Resize
  Resize(*valid_input_size, always_apply=True),


In [6]:
class TestDataset(Dataset):
    def __init__(self, path, tfms):
        super().__init__()
        self.data = [
            (file.name, open(file, "rb").read())
            for file in Path(path).iterdir()
            if file.is_file()
        ]
        self.tfms = tfms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        filename, img_bytes = self.data[idx]
        img = np.array(Image.open(io.BytesIO(img_bytes)).convert("RGB"))
        img = self.tfms(image=img)["image"]
        return filename, img


test_ds = TestDataset(test_img_path, test_tfms)
test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=0, drop_last=False)




In [7]:
class EnsemblePredictor:
    def __init__(self, models):
        self.models = models

    def predict_on_loader(self, loader):
        predictions = []
        filenames = []

        for file_batch, img_batch in loader:
            img_batch = img_batch.to(device)

            # generate flipped variants
            img_h = torch.flip(img_batch, dims=[-1])
            img_v = torch.flip(img_batch, dims=[-2])
            img_hv = torch.flip(img_h, dims=[-2])

            batch_probs = None
            for model in self.models:
                with torch.no_grad():
                    logits_orig = model(img_batch)
                    logits_h = model(img_h)
                    logits_v = model(img_v)
                    logits_hv = model(img_hv)
                    # convert logits to probabilities and sum
                    probs = (
                        F.softmax(logits_orig, dim=1)
                        + F.softmax(logits_h, dim=1)
                        + F.softmax(logits_v, dim=1)
                        + F.softmax(logits_hv, dim=1)
                    )
                if batch_probs is None:
                    batch_probs = probs
                else:
                    batch_probs += probs
            # average over models and augmentations
            batch_preds = torch.argmax(batch_probs, dim=1).cpu().numpy()
            predictions.extend(batch_preds.tolist())
            filenames.extend(file_batch)

        return predictions, filenames


predictor = EnsemblePredictor(models_list)
predictions, filenames = predictor.predict_on_loader(test_loader)



In [8]:
submission_path = "submission.csv"
with open(submission_path, "w") as submission:
    submission.write("image_id,label\n")
    for filename, pred in zip(filenames, predictions):
        submission.write(f"{filename},{pred}\n")
print(f"Submission written to {submission_path}")

Submission written to submission.csv
